## Setup and Installation

In [ ]:
# Install required packages
!pip install transformers datasets evaluate sacrebleu sacremoses sentencepiece torch accelerate peft

## Imports

In [ ]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    GenerationConfig
)
from datasets import Dataset, DatasetDict
from sacremoses import MosesPunctNormalizer
import evaluate
import numpy as np
import torch
import pandas as pd
from pathlib import Path
import json
import re
import sys
import unicodedata
from collections import Counter
from tqdm.auto import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

## Step 1: Looking at the Data

**From article**: "You need to make sure that the training texts do contain many diverse sentences, and that they look more or less clean and consistent."

Key checks:
- Sentence vs word/phrase distribution
- Train/dev/test splits
- Data quality and consistency

In [ ]:
# Configuration
DATA_DIR = Path("../data/splits")
PAIR_NAME = "en-war"  # Change this to your language pair

def load_parallel_data(pair_name):
    """Load parallel corpus from splits directory."""
    src_code, tgt_code = pair_name.split("-")
    pair_dir = DATA_DIR / pair_name
    
    data = {}
    for split in ['train', 'dev', 'test']:
        src_file = pair_dir / f"{split}.{src_code}"
        tgt_file = pair_dir / f"{split}.{tgt_code}"
        
        if src_file.exists() and tgt_file.exists():
            with open(src_file, 'r', encoding='utf-8') as f:
                src_texts = [line.strip() for line in f]
            with open(tgt_file, 'r', encoding='utf-8') as f:
                tgt_texts = [line.strip() for line in f]
            
            data[split] = {
                'source': src_texts,
                'target': tgt_texts,
                'count': len(src_texts)
            }
    
    return data

# Load data
data = load_parallel_data(PAIR_NAME)

# Display statistics
print(f"\n{'='*60}")
print(f"Data Statistics for {PAIR_NAME}")
print(f"{'='*60}")
for split, info in data.items():
    print(f"{split.capitalize():6s}: {info['count']:,} pairs")
    
# Sample data
print(f"\n{'='*60}")
print("Sample Parallel Sentences (first 5 from train)")
print(f"{'='*60}")
for i in range(min(5, len(data['train']['source']))):
    print(f"\n[{i+1}]")
    print(f"  SRC: {data['train']['source'][i]}")
    print(f"  TGT: {data['train']['target'][i]}")

## Step 2: Text Preprocessing (NLLB Pipeline)

**From article**: "The NLLB team preprocessed their texts before training the tokenizer and the model."

Preprocessing steps:
1. Normalize punctuation (Moses)
2. Remove non-printing characters
3. Normalize whitespace

**Impact**: Eliminates `<unk>` tokens from non-standard characters

In [ ]:
# Initialize Moses Punctuation Normalizer
mpn = MosesPunctNormalizer(lang="en")
mpn.substitutions = [(re.compile(r), sub) for r, sub in mpn.substitutions]

def get_non_printing_char_replacer(replace_by: str = " "):
    """
    Create a mapping to replace non-printing characters.
    From: https://github.com/facebookresearch/stopes/blob/main/stopes/pipelines/monolingual/monolingual_line_processor.py#L214
    """
    non_printable_map = {
        ord(c): replace_by
        for c in (chr(i) for i in range(sys.maxunicode + 1))
        # same as \p{C} in perl
        # see https://www.unicode.org/reports/tr44/#General_Category_Values
        if unicodedata.category(c) in {"C", "Cc", "Cf", "Cs", "Co", "Cn"}
    }
    return non_printable_map

non_printable_map = get_non_printing_char_replacer(" ")

def preprocess_text(text):
    """
    Preprocess text following NLLB guidelines.
    This is CRITICAL for avoiding <unk> tokens.
    """
    # Normalize punctuation
    clean = mpn.normalize(text)
    # Remove non-printing chars
    clean = clean.translate(non_printable_map)
    # Normalize whitespace
    clean = " ".join(clean.split())
    return clean

# Test preprocessing
test_text = data['train']['source'][0]
print("Original:")
print(f"  {test_text}")
print("\nPreprocessed:")
print(f"  {preprocess_text(test_text)}")

print("\n✓ Preprocessing functions ready")

## Step 3: Tokenization Analysis

**From article**: "The quality of translation critically depends on how well the tokenizer represents our languages."

Key metrics:
- **Tokens per word**: Target 2-3 for morphologically rich languages
- **`<unk>` frequency**: Should be near 0% after preprocessing

**Article quote**: "For Tyvan, a new language, the NLLB tokenizer produces on average 2.4 tokens per word; almost as few as 2.0 for the well-supported Russian language. This implies that the translation quality of fine-tuned NLLB may be decent without extending its vocabulary."

In [ ]:
# Load NLLB tokenizer and model
MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# NLLB language codes
LANG_CODES = {
    "en": "eng_Latn",
    "war": "war_Latn",
    "ceb": "ceb_Latn",
    "tl": "tgl_Latn"
}

src_code, tgt_code = PAIR_NAME.split("-")
src_lang = LANG_CODES[src_code]
tgt_lang = LANG_CODES[tgt_code]

print(f"Language codes: {src_code} → {src_lang}, {tgt_code} → {tgt_lang}")

def word_tokenize(text):
    """Simple word tokenization for space-separated languages."""
    return re.findall(r'(\w+|[^\w\s])', text)

def analyze_tokenization(texts, lang_code, lang_name, sample_size=1000):
    """
    Analyze tokenization quality following article methodology.
    """
    print(f"\n{'='*60}")
    print(f"Tokenization Analysis: {lang_name} ({lang_code})")
    print(f"{'='*60}")
    
    # Sample texts
    sample_texts = texts[:min(sample_size, len(texts))]
    
    # Preprocess
    preprocessed = [preprocess_text(t) for t in sample_texts]
    
    # Tokenize
    tokenizer.src_lang = lang_code
    tokenized = [tokenizer.tokenize(t) for t in preprocessed]
    words = [word_tokenize(t) for t in preprocessed]
    
    # Calculate statistics
    total_tokens = sum(len(t) for t in tokenized)
    total_words = sum(len(w) for w in words)
    tokens_per_word = total_tokens / total_words if total_words > 0 else 0
    
    # Check <unk> frequency
    texts_with_unk = sum(1 for t in preprocessed 
                         if tokenizer.unk_token_id in tokenizer(t).input_ids)
    unk_percentage = 100 * texts_with_unk / len(preprocessed)
    
    print(f"Sample size: {len(sample_texts):,}")
    print(f"Total words: {total_words:,}")
    print(f"Total tokens: {total_tokens:,}")
    print(f"\n📊 Tokens per word: {tokens_per_word:.2f}")
    print(f"   Target: 2-3 for morphologically rich languages")
    print(f"   Status: {'✓ Good' if 1.5 <= tokens_per_word <= 4 else '⚠️ Check vocabulary'}")
    
    print(f"\n📊 <unk> token frequency: {unk_percentage:.2f}%")
    print(f"   Texts with <unk>: {texts_with_unk:,}/{len(preprocessed):,}")
    print(f"   Status: {'✓ Excellent' if unk_percentage < 1 else '⚠️ Consider vocabulary expansion'}")
    
    # Show sample tokenization
    print(f"\n📝 Sample tokenization:")
    for i in range(min(3, len(preprocessed))):
        print(f"\n  Text: {preprocessed[i][:80]}{'...' if len(preprocessed[i]) > 80 else ''}")
        print(f"  Tokens: {' | '.join(tokenized[i][:15])}{'...' if len(tokenized[i]) > 15 else ''}")
        print(f"  Word count: {len(words[i])}, Token count: {len(tokenized[i])}")
    
    return {
        'tokens_per_word': tokens_per_word,
        'unk_percentage': unk_percentage,
        'total_tokens': total_tokens,
        'total_words': total_words
    }

# Analyze both languages
src_stats = analyze_tokenization(data['train']['source'], src_lang, src_code.upper())
tgt_stats = analyze_tokenization(data['train']['target'], tgt_lang, tgt_code.upper())

## Step 4: Proper Generation Configuration

**From article**: The article demonstrates correct translation with `forced_bos_token_id`:

```python
translated_tokens = model.generate(
    **inputs, 
    forced_bos_token_id=tokenizer.lang_code_to_id["eng_Latn"]
)
```

**Critical**: Without `forced_bos_token_id`, NLLB may generate in the wrong language!

**Note**: Use `tokenizer.convert_tokens_to_ids(lang_code)` instead of `tokenizer.lang_code_to_id` which doesn't exist in the standard API.

In [ ]:
# Load model
print("Loading NLLB-200 model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if torch.cuda.is_available():
    model = model.cuda()
print("✓ Model loaded")

def translate(
    text,
    src_lang,
    tgt_lang,
    max_length=128,
    num_beams=4,
    **kwargs
):
    """
    Translate text using NLLB with proper generation config.
    
    From article: "num_beams: increasing this number usually improves 
    the accuracy, but makes the translation slower."
    """
    # Preprocess
    text = preprocess_text(text)
    
    # Tokenize
    tokenizer.src_lang = src_lang
    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=max_length
    )
    
    if torch.cuda.is_available():
        inputs = inputs.to('cuda')
    
    # Generate with forced target language
    # CRITICAL: forced_bos_token_id ensures correct output language
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    
    model.eval()
    with torch.no_grad():
        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=max_length,
            num_beams=num_beams,
            **kwargs
        )
    
    # Decode
    translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return translation

# Test translation (with pretrained model)
print("\n" + "="*60)
print("Translation Test (Pretrained NLLB-200)")
print("="*60)

test_text = data['train']['source'][0]
reference = data['train']['target'][0]

print(f"\nSource ({src_code}): {test_text}")
print(f"Reference ({tgt_code}): {reference}")

translation = translate(test_text, src_lang, tgt_lang)
print(f"Translation (NLLB): {translation}")

print("\n💡 Note: This is the pretrained model. After fine-tuning, quality should improve significantly.")

## Step 5: Training Configuration

**From article**: Key training parameters:

```python
optimizer = Adafactor(
    scale_parameter=False,
    relative_step=False,
    lr=1e-4,
    clip_threshold=1.0,
    weight_decay=1e-3,
)
scheduler = get_constant_schedule_with_warmup(optimizer, num_warmup_steps=1000)
```

**Key points**:
- Adafactor optimizer saves GPU memory vs AdamW
- Linear warmup for first 1000 steps
- Gradient clipping for stability
- Weight decay prevents overfitting

In [ ]:
# Training configuration following article best practices
MAX_LENGTH = 128
BATCH_SIZE = 4  # Article uses 16 for 15GB GPU
LEARNING_RATE = 1e-4  # Article recommendation
NUM_EPOCHS = 3
WARMUP_STEPS = 1000
NUM_BEAMS = 4

# BF16 Optimization
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

OUTPUT_DIR = Path("../models/nllb_finetuned_best_practices")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Training Configuration (from article):")
print(f"  Model: {MODEL_NAME}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max length: {MAX_LENGTH}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"  Beam search: {NUM_BEAMS} beams")
print(f"  BF16 enabled: {USE_BF16}")
print(f"  Output: {OUTPUT_DIR}")

## Step 6: Prepare Dataset for Training

**From article**: Apply preprocessing before tokenization to eliminate `<unk>` tokens.

In [ ]:
def create_dataset(data_dict, apply_preprocessing=True):
    """Create HuggingFace dataset with preprocessing."""
    datasets = {}
    
    for split, info in data_dict.items():
        src_texts = info['source']
        tgt_texts = info['target']
        
        if apply_preprocessing:
            src_texts = [preprocess_text(t) for t in src_texts]
            tgt_texts = [preprocess_text(t) for t in tgt_texts]
        
        datasets[split] = Dataset.from_dict({
            'source': src_texts,
            'target': tgt_texts
        })
    
    # Map to validation key for HF Trainer
    if 'dev' in datasets:
        datasets['validation'] = datasets.pop('dev')
    
    return DatasetDict(datasets)

def preprocess_function(examples):
    """Tokenize examples for NLLB training."""
    # Set source language
    tokenizer.src_lang = src_lang
    inputs = tokenizer(
        examples['source'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    # Set target language
    tokenizer.tgt_lang = tgt_lang
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples['target'],
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length'
        )
    
    inputs['labels'] = labels['input_ids']
    return inputs

# Create and tokenize dataset
print("Creating dataset with preprocessing...")
dataset = create_dataset(data, apply_preprocessing=True)
print("\nTokenizing dataset...")
tokenized_dataset = dataset.map(preprocess_function, batched=True)
print("✓ Dataset ready for training")
print(f"  Train: {len(tokenized_dataset['train']):,} examples")
print(f"  Validation: {len(tokenized_dataset['validation']):,} examples")

## Step 7: Training Arguments with Critical Parameters

**Critical additions** (from our analysis):
- `generation_max_length`: Required for `predict_with_generate=True`
- GenerationConfig with `forced_bos_token_id`: Ensures correct target language
- `bf16`: BFloat16 precision for better numerical stability (Ampere+ GPUs)

**From article**: The article uses a custom training loop, but we use HF Trainer for simplicity.

In [ ]:
def compute_metrics(eval_pred):
    """Compute BLEU and ChrF++ metrics."""
    bleu = evaluate.load("sacrebleu")
    
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]
    
    # Replace -100 with pad_token_id
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    
    # Decode
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Compute BLEU
    result = bleu.compute(
        predictions=decoded_preds,
        references=[[label] for label in decoded_labels]
    )
    
    return {"bleu": result["score"]}

# Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    weight_decay=1e-3,  # From article
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,  # ⚠️ CRITICAL
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    bf16=USE_BF16,  
    fp16=False, 
    report_to="none"
)

# Configure generation (CRITICAL for NLLB)
forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
gen_config = GenerationConfig(
    max_length=MAX_LENGTH,
    num_beams=NUM_BEAMS,
    forced_bos_token_id=forced_bos_token_id  # ⚠️ CRITICAL
)
model.generation_config = gen_config

print("✓ Training arguments configured")
print(f"  Using BF16: {USE_BF16}")
print(f"  Forced BOS token: {tgt_lang} (id={forced_bos_token_id})")
print(f"  This ensures generation in {tgt_code.upper()}")

## Step 8: Train the Model - Two Approaches

**From article**: "I usually run the training on Google Colab for at most 24 hours."

We'll test two training strategies:
1. **Baseline**: Direct en→war training (3 epochs)
2. **Experimental**: Sequential training - en→ceb first (2 epochs), then en→war (3 epochs)

This tests whether warming up on a similar language (Cebuano) improves performance on the target language (Waray).

In [ ]:
# ============================================================================
# APPROACH 1: BASELINE - Direct en→war Training
# ============================================================================

print("\n" + "="*80)
print("APPROACH 1: BASELINE - Direct en→war Training")
print("="*80)

# Create baseline trainer
baseline_output_dir = OUTPUT_DIR / "baseline_en_war"
baseline_output_dir.mkdir(parents=True, exist_ok=True)

baseline_args = Seq2SeqTrainingArguments(
    output_dir=str(baseline_output_dir),
    eval_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    weight_decay=1e-3,
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    bf16=USE_BF16,
    fp16=False,
    report_to="none"
)

# Load fresh model for baseline
print("\nLoading fresh NLLB model for baseline training...")
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if torch.cuda.is_available():
    baseline_model = baseline_model.cuda()

# Configure generation
forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
baseline_gen_config = GenerationConfig(
    max_length=MAX_LENGTH,
    num_beams=NUM_BEAMS,
    forced_bos_token_id=forced_bos_token_id
)
baseline_model.generation_config = baseline_gen_config

baseline_trainer = Seq2SeqTrainer(
    model=baseline_model,
    args=baseline_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print(f"\nStarting baseline training: en→war for {NUM_EPOCHS} epochs...")
baseline_result = trainer.train()

# Save baseline model
baseline_final_dir = baseline_output_dir / "final_model"
baseline_trainer.save_model(str(baseline_final_dir))
tokenizer.save_pretrained(str(baseline_final_dir))

print("\n" + "="*80)
print("Baseline Training Complete")
print("="*80)
print(f"  Training loss: {baseline_result.training_loss:.4f}")
print(f"  Training time: {baseline_result.metrics['train_runtime']:.1f}s")
print(f"  Model saved to: {baseline_final_dir}")

# Evaluate baseline
print("\nEvaluating baseline model...")
baseline_eval = baseline_trainer.evaluate()
print(f"  Baseline BLEU: {baseline_eval['eval_bleu']:.2f}")

In [ ]:
# ============================================================================
# APPROACH 2: EXPERIMENTAL - Sequential Training (en→ceb, then en→war)
# ============================================================================

print("\n\n" + "="*80)
print("APPROACH 2: EXPERIMENTAL - Sequential Training")
print("="*80)

# ------------------------------------------------------------------------------
# Stage 1: Train on en→ceb
# ------------------------------------------------------------------------------

print("\n" + "-"*80)
print("Stage 1: Training en→ceb (Similar Language)")
print("-"*80)

# Load en-ceb data
ceb_data = load_parallel_data("en-ceb")
ceb_dataset = create_dataset(ceb_data, apply_preprocessing=True)

# Update language codes for Cebuano
ceb_tgt_lang = LANG_CODES["ceb"]

def preprocess_function_ceb(examples):
    """Tokenize examples for en→ceb training."""
    tokenizer.src_lang = src_lang
    inputs = tokenizer(
        examples['source'],
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    tokenizer.tgt_lang = ceb_tgt_lang
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples['target'],
            max_length=MAX_LENGTH,
            truncation=True,
            padding='max_length'
        )
    
    inputs['labels'] = labels['input_ids']
    return inputs

print("Tokenizing en→ceb dataset...")
ceb_tokenized = ceb_dataset.map(preprocess_function_ceb, batched=True)
print(f"  Train: {len(ceb_tokenized['train']):,} examples")
print(f"  Validation: {len(ceb_tokenized['validation']):,} examples")

# Stage 1 training configuration
stage1_output_dir = OUTPUT_DIR / "experimental_stage1_en_ceb"
stage1_output_dir.mkdir(parents=True, exist_ok=True)

stage1_args = Seq2SeqTrainingArguments(
    output_dir=str(stage1_output_dir),
    eval_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=2,  # 2 epochs for Stage 1
    warmup_steps=WARMUP_STEPS,
    weight_decay=1e-3,
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    bf16=USE_BF16,
    fp16=False,
    report_to="none"
)

# Load fresh model for Stage 1
print("\nLoading fresh NLLB model for Stage 1...")
stage1_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
if torch.cuda.is_available():
    stage1_model = stage1_model.cuda()

# Configure generation for Cebuano
forced_bos_ceb = tokenizer.convert_tokens_to_ids(ceb_tgt_lang)
stage1_gen_config = GenerationConfig(
    max_length=MAX_LENGTH,
    num_beams=NUM_BEAMS,
    forced_bos_token_id=forced_bos_ceb
)
stage1_model.generation_config = stage1_gen_config

stage1_trainer = Seq2SeqTrainer(
    model=stage1_model,
    args=stage1_args,
    train_dataset=ceb_tokenized['train'],
    eval_dataset=ceb_tokenized['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("\nStarting Stage 1 training: en→ceb for 2 epochs...")
stage1_result = stage1_trainer.train()

# Save Stage 1 model
stage1_final_dir = stage1_output_dir / "final_model"
stage1_trainer.save_model(str(stage1_final_dir))
tokenizer.save_pretrained(str(stage1_final_dir))

print("\n" + "-"*80)
print("Stage 1 Complete")
print("-"*80)
print(f"  Training loss: {stage1_result.training_loss:.4f}")
print(f"  Training time: {stage1_result.metrics['train_runtime']:.1f}s")
print(f"  Model saved to: {stage1_final_dir}")

stage1_eval = stage1_trainer.evaluate()
print(f"  Stage 1 BLEU (en→ceb): {stage1_eval['eval_bleu']:.2f}")

# ------------------------------------------------------------------------------
# Stage 2: Fine-tune Stage 1 model on en→war
# ------------------------------------------------------------------------------

print("\n" + "-"*80)
print("Stage 2: Fine-tuning on en→war (Target Language)")
print("-"*80)

# Stage 2 training configuration
stage2_output_dir = OUTPUT_DIR / "experimental_stage2_en_war"
stage2_output_dir.mkdir(parents=True, exist_ok=True)

stage2_args = Seq2SeqTrainingArguments(
    output_dir=str(stage2_output_dir),
    eval_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,  # 3 epochs for Stage 2
    warmup_steps=WARMUP_STEPS,
    weight_decay=1e-3,
    save_strategy="epoch",
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_LENGTH,
    logging_strategy="steps",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    bf16=USE_BF16,
    fp16=False,
    report_to="none"
)

# Load Stage 1 model for continued training
print(f"\nLoading Stage 1 model from: {stage1_final_dir}")
stage2_model = AutoModelForSeq2SeqLM.from_pretrained(str(stage1_final_dir))
if torch.cuda.is_available():
    stage2_model = stage2_model.cuda()

# Configure generation for Waray
forced_bos_war = tokenizer.convert_tokens_to_ids(tgt_lang)
stage2_gen_config = GenerationConfig(
    max_length=MAX_LENGTH,
    num_beams=NUM_BEAMS,
    forced_bos_token_id=forced_bos_war
)
stage2_model.generation_config = stage2_gen_config

stage2_trainer = Seq2SeqTrainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=tokenized_dataset['train'],  # en→war dataset
    eval_dataset=tokenized_dataset['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print(f"\nStarting Stage 2 training: en→war for {NUM_EPOCHS} epochs...")
print("  (continuing from en→ceb model)")
stage2_result = stage2_trainer.train()

# Save Stage 2 model
stage2_final_dir = stage2_output_dir / "final_model"
stage2_trainer.save_model(str(stage2_final_dir))
tokenizer.save_pretrained(str(stage2_final_dir))

print("\n" + "-"*80)
print("Stage 2 Complete")
print("-"*80)
print(f"  Training loss: {stage2_result.training_loss:.4f}")
print(f"  Training time: {stage2_result.metrics['train_runtime']:.1f}s")
print(f"  Model saved to: {stage2_final_dir}")

# Evaluate Stage 2
print("\nEvaluating experimental model...")
stage2_eval = stage2_trainer.evaluate()
print(f"  Experimental BLEU (en→war after en→ceb): {stage2_eval['eval_bleu']:.2f}")

# ------------------------------------------------------------------------------
# Compare Results
# ------------------------------------------------------------------------------

print("\n\n" + "="*80)
print("FINAL COMPARISON")
print("="*80)
print(f"\nBaseline (Direct en→war):          {baseline_eval['eval_bleu']:.2f} BLEU")
print(f"Experimental (en→ceb → en→war):    {stage2_eval['eval_bleu']:.2f} BLEU")

improvement = stage2_eval['eval_bleu'] - baseline_eval['eval_bleu']
print(f"\nImprovement: {improvement:+.2f} BLEU points")

if improvement > 0:
    print("✓ Sequential training through similar language IMPROVED performance")
elif improvement < 0:
    print("✗ Sequential training through similar language DEGRADED performance")
else:
    print("= No difference in performance")

# Save comparison results
comparison_results = {
    "baseline": {
        "approach": "Direct en→war",
        "epochs": NUM_EPOCHS,
        "bleu": baseline_eval['eval_bleu'],
        "training_loss": baseline_result.training_loss,
        "training_time": baseline_result.metrics['train_runtime'],
        "model_path": str(baseline_final_dir)
    },
    "experimental": {
        "approach": "Sequential: en→ceb (2 epochs) → en→war (3 epochs)",
        "stage1": {
            "pair": "en→ceb",
            "epochs": 2,
            "bleu": stage1_eval['eval_bleu'],
            "training_loss": stage1_result.training_loss,
            "training_time": stage1_result.metrics['train_runtime']
        },
        "stage2": {
            "pair": "en→war",
            "epochs": NUM_EPOCHS,
            "bleu": stage2_eval['eval_bleu'],
            "training_loss": stage2_result.training_loss,
            "training_time": stage2_result.metrics['train_runtime']
        },
        "model_path": str(stage2_final_dir)
    },
    "improvement": improvement
}

comparison_file = OUTPUT_DIR / "training_comparison.json"
with open(comparison_file, 'w') as f:
    json.dump(comparison_results, f, indent=2)

print(f"\nComparison results saved to: {comparison_file}")

# Set final_model_dir to the better performing model for evaluation
if improvement >= 0:
    final_model_dir = stage2_final_dir
    print(f"\n✓ Using experimental model for final evaluation")
else:
    final_model_dir = baseline_final_dir
    print(f"\n✓ Using baseline model for final evaluation")

## Step 9: Evaluate with BLEU and ChrF++

**From article**: "The two most popular automatic metrics for machine translation quality are BLEU and ChrF++."

- **BLEU**: Word-level precision (full matches only)
- **ChrF++**: Character-level (better for morphologically rich languages)

**Important**: "Comparing their values for different target languages (or even for the same language, but using different data) is not so meaningful."

In [ ]:
import sacrebleu

# Load fine-tuned model
print("Loading fine-tuned model...")
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(str(final_model_dir))
finetuned_tokenizer = AutoTokenizer.from_pretrained(str(final_model_dir))
if torch.cuda.is_available():
    finetuned_model = finetuned_model.cuda()
print("✓ Model loaded")

def translate_batch(texts, model, tokenizer, src_lang, tgt_lang, batch_size=16):
    """
    Translate texts in batches (from article optimization).
    """
    # Preprocess
    texts = [preprocess_text(t) for t in texts]
    
    # Sort by length for efficient batching
    idxs, sorted_texts = zip(*sorted(enumerate(texts), key=lambda p: len(p[1]), reverse=True))
    
    translations = []
    model.eval()
    
    for i in tqdm(range(0, len(sorted_texts), batch_size), desc="Translating"):
        batch = sorted_texts[i:i+batch_size]
        
        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH
        )
        
        if torch.cuda.is_available():
            inputs = inputs.to('cuda')
        
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=MAX_LENGTH,
                num_beams=NUM_BEAMS
            )
        
        batch_translations = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        translations.extend(batch_translations)
    
    # Restore original order
    return [t for _, t in sorted(zip(idxs, translations))]

# Evaluate on test set
print("\n" + "="*60)
print("Evaluation on Test Set")
print("="*60)

test_sources = data['test']['source'] if 'test' in data else data['validation']['source'][:100]
test_references = data['test']['target'] if 'test' in data else data['validation']['target'][:100]

print(f"Translating {len(test_sources)} test examples...")
predictions = translate_batch(
    test_sources,
    finetuned_model,
    finetuned_tokenizer,
    src_lang,
    tgt_lang
)

# Compute metrics
bleu_calc = sacrebleu.BLEU()
chrf_calc = sacrebleu.CHRF(word_order=2)  # ChrF++

# Preprocess references for fair comparison
test_references = [preprocess_text(r) for r in test_references]

bleu_score = bleu_calc.corpus_score(predictions, [test_references])
chrf_score = chrf_calc.corpus_score(predictions, [test_references])

print(f"\n{'='*60}")
print("Results")
print(f"{'='*60}")
print(f"BLEU: {bleu_score.score:.2f}")
print(f"ChrF++: {chrf_score.score:.2f}")
print(f"\nTest set size: {len(test_sources)} pairs")

# Show some examples
print(f"\n{'='*60}")
print("Sample Translations")
print(f"{'='*60}")
for i in range(min(5, len(test_sources))):
    print(f"\n[{i+1}]")
    print(f"  Source: {test_sources[i]}")
    print(f"  Reference: {test_references[i]}")
    print(f"  Translation: {predictions[i]}")

## Key Takeaways from Article

### ✅ Must-Do:
1. **Preprocess all texts** before training (Moses + non-printing char removal)
2. **Analyze tokenization** - check tokens/word and `<unk>` frequency
3. **Use `forced_bos_token_id`** - critical for correct target language
4. **Split long texts** into sentences before translation
5. **Batch by length** for efficient GPU usage
6. **Use beam search** (`num_beams=4`) for better quality

### 📊 Evaluation:
- Use both BLEU and ChrF++
- Compare models on same dataset only
- ChrF++ better for morphologically rich languages

### ⚠️ Common Issues:
- **Wrong language output**: Missing `forced_bos_token_id`
- **Incomplete translations**: Input text too long (split into sentences!)
- **Poor quality**: High `<unk>` frequency (add preprocessing or expand vocabulary)
- **Slow training**: Use Adafactor optimizer to save memory

### 📚 Reference:
[How to fine-tune a NLLB-200 model for translating a new language](https://cointegrated.medium.com/how-to-fine-tune-a-nllb-200-model-for-translating-a-new-language-a37fc706b865)  
by David Dale, Meta AI Research